In [1]:
import numpy as np
from pyscf import gto, scf, cc

####  test H2 monomers ####
a = 2 # bond length in a cluster
d = 4 # distance between each cluster
unit = 'b' # unit of length
na = 2 # size of a cluster (monomer)
nc = 1 # set as integer multiple of monomers
spin = 2 # spin per monomer
frozen = 0 # frozen orbital per monomer
elmt = 'O'
unit = 'B'
basis = 'sto6g'
atoms = ""
for n in range(nc*na):
    shift = ((n - n % na) // na) * (d-a)
    atoms += f"{elmt} {n*a+shift:.5f} 0.00000 0.00000 \n"
###########################

mol = gto.M(atom=atoms,
            basis="sto6g",
            verbose=4,
            unit=unit,
            symmetry=0,
            charge=0,
            spin=spin*nc,
            max_memory=40000,
            )

mf = scf.RHF(mol)
mf.kernel()

mycc = cc.CCSD(mf).set_frozen()
mycc.kernel()

System: uname_result(system='Linux', node='sharmagroup-rn', release='6.17.0-35-generic', version='#35~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Tue May 26 19:30:42 UTC 2', machine='x86_64')  Threads 16
Python 3.12.13 | packaged by Anaconda, Inc. | (main, Mar 19 2026, 20:20:58) [GCC 14.3.0]
numpy 2.4.4  scipy 1.17.1  h5py 3.16.0
Date: Wed Jun  3 15:01:19 2026
PySCF version 2.12.1
PySCF path  /home/sharmagroup/sharmagroup/pyscf
GIT ORIG_HEAD 3d1768f5e33b144b606c3d2c81c12ee54d794501
GIT HEAD (branch master) f0861da51f017364d8bbaa20b742a94f3733305f

[ENV] OLD_PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:
[ENV] PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:/home/sharmagroup/sharmagroup/pyscf-forge:
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 2
[INPUT] num. electrons = 16
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 2
[INPUT] symmetry 0 subgroup None
[INPUT] Mole.unit = B
[INPUT] Symbol           X                Y                Z      uni

(np.float64(-0.07765774343540227),
 (array([[-1.28538440e-15],
         [ 3.32812353e-02],
         [-5.86348049e-17],
         [ 1.68063665e-16],
         [ 1.54547239e-15],
         [ 2.05849151e-16],
         [ 1.77133392e-16]]),
  array([[ 2.11911985e-16, -7.33671522e-17, -1.28330229e-15],
         [-6.00744002e-16,  6.42485014e-17, -1.59591218e-02],
         [-9.94817927e-16, -6.37688442e-16, -1.36569265e-16],
         [ 1.97692132e-15,  1.31189248e-15,  1.01888432e-16],
         [ 5.01548293e-16,  5.05446521e-16, -1.07655515e-15]])),
 (array([[[[0.]],
  
          [[0.]],
  
          [[0.]],
  
          [[0.]],
  
          [[0.]],
  
          [[0.]],
  
          [[0.]]],
  
  
         [[[0.]],
  
          [[0.]],
  
          [[0.]],
  
          [[0.]],
  
          [[0.]],
  
          [[0.]],
  
          [[0.]]],
  
  
         [[[0.]],
  
          [[0.]],
  
          [[0.]],
  
          [[0.]],
  
          [[0.]],
  
          [[0.]],
  
          [[0.]]],
  
  
 

In [2]:
import jax
jax.config.update("jax_enable_x64", True)

from afqmc import integral
integral.prep_integral(mycc)


Preparing AFQMC calculation
Calculating Cholesky integrals
Alpha Cholesky shape: (42, 10, 10) 
 Beta Cholesky shape: (42, 10, 10) 
Finished calculating Cholesky integrals
Size of the correlation space:
Number of electrons:        [7 5]
Number of basis functions:  8
Number of Cholesky vectors: 42


In [13]:
from jax import random
from jax import numpy as jnp
from afqmc import config, prep

config.setup_jax()

options = {'eql_time': 10,
            'n_blocks': 100,
            'n_walkers': 300,
            'max_error': 0.0,
            'nchol_chunk': 30,
            'max_memory': 3000,
            'seed': 17,
            'walker_type': 'uhf',
            'trial': 'upt2ccsd',
            }

ham_data, ham, prop, trial, wave_data, sampler, options = (prep.init_afqmc(options=options))
wave_data["rdm1"] = trial.get_rdm1(wave_data)
ham_data = ham.build_measurement_intermediates(ham_data, trial, wave_data)
ham_data = ham.build_propagation_intermediates(ham_data, prop, trial, wave_data)

Hostname:     sharmagroup-rn
System:       Linux
Node:         sharmagroup-rn
Release:      6.17.0-35-generic
Machine:      x86_64
Processor:    x86_64
JAX backend:  GPU
JAX devices:  [CudaDevice(id=0)]
Device kind:  NVIDIA GeForce RTX 5060 Ti
Platform:     gpu

Load system from Integral File
Maximum memory per walker:            10.00 MB
Maximum number of Cholesky per chunk: 5120
Number of Cholesky chunks:            1
Number of Cholesky per chunk:         42
Number of padding Cholesky:           0

QMC System
Number of electrons: (7, 5)
Spin Multiplicity:   2
Number of orbitals:  8
Number of Chol:      42

QMC Parameters
eql_time            :                   10
n_blocks            :                  100
n_walkers           :                  300
max_error           :                  0.0
nchol_chunk         :                   42
max_memory          :                 3000
seed                :                   17
walker_type         :                  uhf
trial               :    

In [28]:
from afqmc import linalg_utils

def init_ccsd_prop_data1(
    trial,
    wave_data,
    ham_data,
    options,
):
    prop_data = {}
    prop_data["weights"] = jnp.ones(prop.n_walkers)
    prop_data["key"] = random.PRNGKey(options["seed"])

    print("\nInitalize QMC walkers by stochastic CCSD")
    init_walkers, prop_data = prep.get_ccsd_walkers(
        prop_data, wave_data, options["n_walkers"], options["walker_type"]
    )
    prop_data["walkers"] = init_walkers

    h0 = ham_data["h0"]
    t1s, t2s, e0s, e1s = trial.calc_energy_pt(prop_data["walkers"], ham_data, wave_data)
    olps = trial.calc_overlap(prop_data["walkers"], wave_data)
    prop_data["overlaps"] = olps

    olp = jnp.sum(olps)
    t1 = jnp.sum(olps * t1s) / olp
    t2 = jnp.sum(olps * t2s) / olp
    e0 = jnp.sum(olps * e0s) / olp
    e1 = jnp.sum(olps * e1s) / olp

    energy = jnp.real(h0 + e0 / t1 + e1 / t1 - t2 * e0 / t1**2)

    prop_data["e_estimate"] = energy
    prop_data["pop_control_ene_shift"] = energy

    return prop_data


def init_ccsd_prop_data2(
    trial,
    wave_data,
    ham_data,
    options,
):

    print("\nInitalize QMC walkers by stochastic CCSD")
    prop_data = {}
    prop_data["n_killed_walkers"] = 0
    prop_data["key"] = random.PRNGKey(options["seed"])

    weights0 = jnp.ones(options["n_walkers"])
    walkers0 = prep.replicate_walker(wave_data["mo_coeff"], options["n_walkers"])
    overlaps0 = trial.calc_overlap(walkers0, wave_data)

    walkers1, prop_data = prep.get_ccsd_walkers(
        prop_data, wave_data, options["n_walkers"], options["walker_type"]
    )
    overlaps1 = trial.calc_overlap(walkers1, wave_data)
    weights1 = jnp.real(weights0 * overlaps1 / overlaps0)

    prop_data["weights"] = weights1
    prop_data["walkers"] = walkers1
    prop_data["overlaps"] = overlaps1

    h0 = ham_data["h0"]
    t1s, t2s, e0s, e1s = trial.calc_energy_pt(prop_data["walkers"], ham_data, wave_data)

    wt = jnp.sum(weights1)
    t1 = jnp.sum(weights1 * t1s) / wt
    t2 = jnp.sum(weights1 * t2s) / wt
    e0 = jnp.sum(weights1 * e0s) / wt
    e1 = jnp.sum(weights1 * e1s) / wt

    energy = jnp.real(h0 + e0 / t1 + e1 / t1 - t2 * e0 / t1**2)

    prop_data["e_estimate"] = energy
    prop_data["pop_control_ene_shift"] = energy

    return prop_data

def init_ccsd_prop_data3(
        trial,
        wave_data, 
        ham_data,
        options,
        ):
    
    print("\nInitalize QMC walkers by stochastic CCSD")
    prop_data = {}
    prop_data["n_killed_walkers"] = 0
    prop_data["key"] = random.PRNGKey(options["seed"])
    
    weights0 = jnp.ones(options["n_walkers"])
    walkers0 = prep.replicate_walker(wave_data["mo_coeff"], options["n_walkers"])
    overlaps0 = trial.calc_overlap(walkers0, wave_data)

    walkers1, prop_data = prep.get_ccsd_walkers(
        prop_data, wave_data, options["n_walkers"], options["walker_type"])
    if isinstance(walkers1, jax.Array):
        walkers1, norms1 = linalg_utils.qr_vmap(walkers1)
    elif isinstance(walkers1, (tuple, list)):
        walkers1, [norms1_a, norms1_b] = linalg_utils.qr_vmap_uhf(walkers1)
        norms1 = norms1_a * norms1_b
        
    overlaps1 = trial.calc_overlap(walkers1, wave_data)
    weights1 = jnp.real(weights0 * overlaps1 / overlaps0)

    prop_data["weights"] = weights1
    prop_data["walkers"] = walkers1
    prop_data["overlaps"] = overlaps1
    prop_data["norms"] = norms1

    h0 = ham_data["h0"]
    t1s, t2s, e0s, e1s = trial.calc_energy_pt(prop_data["walkers"], ham_data, wave_data)

    wt = jnp.sum(weights1)
    t1 = jnp.sum(weights1 * t1s) / wt
    t2 = jnp.sum(weights1 * t2s) / wt
    e0 = jnp.sum(weights1 * e0s) / wt
    e1 = jnp.sum(weights1 * e1s) / wt

    energy = jnp.real(h0 + e0 / t1 + e1 / t1 - t2 * e0 / t1**2)

    prop_data["e_estimate"] = energy
    prop_data["pop_control_ene_shift"] = energy

    return prop_data

In [29]:
prop_data1 = init_ccsd_prop_data1(trial, wave_data, ham_data, options)
prop_data2 = init_ccsd_prop_data2(trial, wave_data, ham_data, options)
prop_data3 = init_ccsd_prop_data3(trial, wave_data, ham_data, options)


Initalize QMC walkers by stochastic CCSD

Initalize QMC walkers by stochastic CCSD

Initalize QMC walkers by stochastic CCSD


In [30]:
print(prop_data1["e_estimate"])
print(prop_data2["e_estimate"])
print(prop_data3["e_estimate"])

-149.04341053267086
-149.04341053267086
-149.04352452059985


In [31]:
energies1 = trial.calc_energy(prop_data1["walkers"], ham_data, wave_data)
energies2 = trial.calc_energy(prop_data2["walkers"], ham_data, wave_data)
energies3 = trial.calc_energy(prop_data3["walkers"], ham_data, wave_data)

In [32]:
overlaps1 = trial.calc_overlap(prop_data1["walkers"], wave_data)
overlaps2 = trial.calc_overlap(prop_data2["walkers"], wave_data)
overlaps3 = trial.calc_overlap(prop_data3["walkers"], wave_data)

In [33]:
n = 1
print(overlaps1[n], energies1[n])
print(overlaps2[n], energies2[n])
print(overlaps3[n], energies3[n])

(1+0j) (-149.10828564936236-0.01235372175913732j)
(1+0j) (-149.10828564936236-0.01235372175913732j)
(0.596190162679378-0j) (-149.10828564936236-0.01235372175913696j)


In [34]:
tot_energy1 = jnp.sum(overlaps1*energies1)/jnp.sum(overlaps1)
tot_energy2 = jnp.sum(overlaps2*energies2)/jnp.sum(overlaps2)
tot_energy3 = jnp.sum(overlaps3*energies3)/jnp.sum(overlaps3)
print(tot_energy1)
print(tot_energy2)
print(tot_energy3)

(-149.04338357414255-0.0005349058192907804j)
(-149.04338357414255-0.0005349058192907804j)
(-149.03844325206603-0.0004832269687478648j)


In [37]:
tot_energy1 = jnp.sum(overlaps1*energies1)/jnp.sum(overlaps1)
tot_energy2 = jnp.sum(overlaps2*energies2)/jnp.sum(overlaps2)
tot_energy3 = jnp.sum(prop_data3["norms"]*overlaps3*energies3)/jnp.sum(prop_data3["norms"]*overlaps3)
print(tot_energy1)
print(tot_energy2)
print(tot_energy3)

(-149.04338357414255-0.0005349058192907804j)
(-149.04338357414255-0.0005349058192907804j)
(-149.04338357414255-0.000534905819290768j)
